[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/10_embeddings_contextuales_oraciones/10_embeddings_contextuales_oraciones.ipynb)
# Módulo 10 · Embeddings contextuales y de oraciones

| | |
|---|---|
| **Nivel** | 🟡 Intermedio |
| **Duración** | ≈ 5 h (lección ≈ 3 h + proyecto ≈ 2 h) |
| **Cómputo** | CPU basta en `FAST_DEV_RUN` (≈ 5 min); con modelos reales, T4 recomendada (≈ 0.5 h de T4: multiplica por la tarifa de unidades/hora que muestra tu panel de recursos de Colab) |
| **Prerrequisitos** | 09 (capa de embedding, anisotropía de $E$), atención y Transformers |
| **Hilo conductor** | Políglota necesita **un vector por documento y por consulta**. Hoy aprendes a sacarlo de un encoder preentrenado y a medir cuál funciona en nuestro corpus |

> **Dónde estamos.** En el 09 cada token tenía **un** vector fijo (fila de $E$). Un Transformer produce un vector por token **en contexto** (capa $\ell$), y para buscar necesitamos comprimir la secuencia en **un** vector: *pooling*. Este módulo compara cómo hacerlo, por qué los vectores «crudos» de BERT funcionan mal para similitud (anisotropía) y cómo se arregla sin entrenar (centrado, *whitening*). Entrenar el encoder para similitud es el módulo 11.

## Objetivos de aprendizaje
1. **Distinguir** embeddings estáticos (fila de $E$) de contextuales (estado oculto de la capa $\ell$) y **medir** la contextualidad por capa (*self-similarity*, Ethayarajh 2019).
2. **Implementar desde cero** pooling CLS, media, máximo y último token con máscaras correctas, y **verificarlo** contra `sentence-transformers`.
3. **Explicar** la arquitectura bi-encoder (SBERT) frente a cross-encoder y su coste.
4. **Cuantificar** la anisotropía y las dimensiones dominantes (*rogue dimensions*) de un encoder.
5. **Aplicar** centrado, *whitening* (Su et al., 2021) y *all-but-the-top*, y **evaluar** su efecto en nDCG@10 sobre Políglota.
6. **Elegir** entre coseno, producto escalar y L2 sabiendo cuándo son equivalentes.
7. **Usar** prefijos/instrucciones (E5: `query:`/`passage:`) y justificar por qué existen.

## 🔁 Conexión con módulos anteriores
<details><summary><b>1.</b> (09) ¿Por qué no sirve usar directamente filas de $E$ promediadas como embedding de una frase?</summary>

Porque $E$ es una tabla *sin contexto* («banco» tiene un solo vector) y además es **anisótropa**: una media común grande domina el coseno. El promedio hereda ambas cosas.
</details>

<details><summary><b>2.</b> (09) En la lección anterior, el coseno medio entre pares aleatorios de filas bajaba a ≈ 0 al **centrar**. ¿Qué conclusión geométrica sacaste?</summary>

Que la anisotropía venía de una **dirección media compartida**: restarla devuelve vectores repartidos alrededor del origen. Hoy aplicamos lo mismo (y más: *whitening*) a embeddings de frases.
</details>

<details><summary><b>3.</b> (00) ¿Qué mide nDCG@10 y por qué lo usamos en Políglota?</summary>

La calidad del top-10 con descuento logarítmico por posición, normalizada por el ranking ideal. Es la métrica estándar de BEIR/MTEB y la del baseline BM25 del módulo 00.
</details>

<details><summary><b>4.</b> (03–04, Diagnóstico) Un encoder WordPiece parte «configuración» en 4 piezas y «setup» en 1. Con *mean pooling*, ¿qué palabra pesa más en el vector de la frase «setup de configuración»?</summary>

«configuración»: aporta 4 de los 6 estados promediados (más [CLS]/[SEP] si se incluyen), «setup» solo 1. La **fertilidad** del módulo 04 no es solo coste: con media, las palabras muy troceadas (y los idiomas con más fertilidad) pesan más en el embedding. Los estados son contextuales y en parte lo compensan, pero conviene saberlo al leer la sección 5.
</details>

In [ ]:
!pip install -q transformers sentence-transformers tokenizers datasets scipy scikit-learn matplotlib
import copy, json, math, os, random, re, time, warnings
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

warnings.filterwarnings("ignore")
import transformers
transformers.logging.set_verbosity_error(); transformers.utils.logging.disable_progress_bar()
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
FAST_DEV_RUN = True   # True: corpus pequeño y modelos diminutos de fallback en minutos de CPU
PAL = {"azul": "#1F6FEB", "naranja": "#F28E2B", "verde": "#2CA02C", "gris": "#6B7280", "morado": "#9467BD"}
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})
print(f"device={device} · FAST_DEV_RUN={FAST_DEV_RUN} · torch {torch.__version__}")

### Utilidades del curso: corpus Políglota y métricas canónicas (módulo 00)

In [ ]:
# ── Datos para el corpus SINTÉTICO de Políglota (último recurso si no hay red / Hugging Face) ──
# (artículo+concepto ES, artículo+concepto EN, definición ES, definición EN)
CONCEPTOS = [
    ("el índice invertido", "the inverted index", "una estructura que asocia cada término con la lista de documentos que lo contienen", "a structure that maps each term to the list of documents that contain it"),
    ("la caché", "the cache", "una memoria rápida que guarda resultados recientes para no recalcularlos", "a fast memory that stores recent results so they are not recomputed"),
    ("la base de datos relacional", "the relational database", "un sistema que organiza los datos en tablas relacionadas mediante claves", "a system that organizes data into tables related through keys"),
    ("el balanceador de carga", "the load balancer", "un componente que reparte las peticiones entre varios servidores", "a component that spreads requests across several servers"),
    ("el contenedor Docker", "the Docker container", "un proceso aislado que empaqueta una aplicación con sus dependencias", "an isolated process that packages an application with its dependencies"),
    ("el control de versiones", "version control", "un sistema que registra los cambios de los archivos a lo largo del tiempo", "a system that records changes to files over time"),
    ("la API REST", "the REST API", "una interfaz que expone recursos mediante URL y verbos HTTP", "an interface that exposes resources through URLs and HTTP verbs"),
    ("la autenticación con tokens", "token authentication", "un mecanismo que identifica al usuario con un testigo firmado en cada petición", "a mechanism that identifies the user with a signed token on every request"),
    ("la expresión regular", "the regular expression", "un patrón que describe un conjunto de cadenas de texto", "a pattern that describes a set of text strings"),
    ("el hilo de ejecución", "the thread", "una secuencia de instrucciones que comparte memoria con otros hilos del proceso", "a sequence of instructions that shares memory with other threads of the process"),
    ("el recolector de basura", "the garbage collector", "un mecanismo que libera automáticamente la memoria que ya no se usa", "a mechanism that automatically frees memory that is no longer used"),
    ("la cola de mensajes", "the message queue", "un búfer que desacopla a los productores de los consumidores de eventos", "a buffer that decouples event producers from consumers"),
    ("la tabla hash", "the hash table", "una estructura que guarda pares clave-valor y busca en tiempo casi constante", "a structure that stores key-value pairs and looks them up in near-constant time"),
    ("el árbol binario de búsqueda", "the binary search tree", "un árbol ordenado donde cada nodo separa los valores menores de los mayores", "an ordered tree where each node separates smaller values from larger ones"),
    ("la red neuronal", "the neural network", "un modelo formado por capas de unidades que aprenden pesos a partir de ejemplos", "a model made of layers of units that learn weights from examples"),
    ("el descenso de gradiente", "gradient descent", "un método que ajusta los parámetros en la dirección que reduce la pérdida", "a method that adjusts parameters in the direction that reduces the loss"),
    ("el sobreajuste", "overfitting", "el fenómeno por el que un modelo memoriza el entrenamiento y generaliza mal", "the phenomenon where a model memorizes the training set and generalizes poorly"),
    ("la validación cruzada", "cross-validation", "una técnica que estima el rendimiento repartiendo los datos en varios pliegues", "a technique that estimates performance by splitting the data into several folds"),
    ("el tokenizador", "the tokenizer", "el componente que convierte el texto en una secuencia de identificadores enteros", "the component that turns text into a sequence of integer identifiers"),
    ("la ventana de contexto", "the context window", "la cantidad máxima de tokens que un modelo puede leer a la vez", "the maximum number of tokens a model can read at once"),
    ("el compilador", "the compiler", "un programa que traduce código fuente a instrucciones de máquina", "a program that translates source code into machine instructions"),
    ("el sistema de archivos", "the file system", "la capa que organiza cómo se guardan y se nombran los datos en disco", "the layer that organizes how data is stored and named on disk"),
    ("la memoria virtual", "virtual memory", "una abstracción que da a cada proceso su propio espacio de direcciones", "an abstraction that gives each process its own address space"),
    ("el protocolo TCP", "the TCP protocol", "un protocolo que entrega bytes de forma ordenada y fiable entre dos máquinas", "a protocol that delivers bytes reliably and in order between two machines"),
    ("el cifrado simétrico", "symmetric encryption", "un esquema que usa la misma clave para cifrar y descifrar", "a scheme that uses the same key to encrypt and decrypt"),
    ("las pruebas unitarias", "unit tests", "comprobaciones automáticas que verifican una función aislada", "automatic checks that verify a single isolated function"),
    ("la integración continua", "continuous integration", "una práctica que compila y prueba cada cambio de forma automática", "a practice that builds and tests every change automatically"),
    ("los microservicios", "microservices", "una arquitectura que divide la aplicación en servicios pequeños e independientes", "an architecture that splits the application into small independent services"),
    ("la paginación", "pagination", "una técnica que devuelve los resultados en bloques de tamaño fijo", "a technique that returns results in fixed-size blocks"),
    ("la limitación de tasa", "rate limiting", "una política que restringe cuántas peticiones puede hacer un cliente por minuto", "a policy that restricts how many requests a client can make per minute"),
    ("la serialización JSON", "JSON serialization", "el proceso de convertir objetos en texto JSON y viceversa", "the process of converting objects to JSON text and back"),
    ("la variable de entorno", "the environment variable", "un par nombre-valor que el sistema operativo entrega a cada proceso", "a name-value pair that the operating system hands to each process"),
    ("el entorno virtual", "the virtual environment", "una carpeta aislada con su propio intérprete de Python y sus paquetes", "an isolated folder with its own Python interpreter and packages"),
    ("el decorador", "the decorator", "una función que envuelve a otra para añadirle comportamiento", "a function that wraps another one to add behavior"),
    ("el generador", "the generator", "una función que produce valores perezosamente con la palabra yield", "a function that lazily produces values with the yield keyword"),
    ("el manejo de excepciones", "exception handling", "el mecanismo para capturar errores sin que el programa se detenga", "the mechanism for catching errors without stopping the program"),
    ("la programación asíncrona", "asynchronous programming", "un modelo que solapa esperas de entrada y salida sin bloquear el hilo", "a model that overlaps input and output waits without blocking the thread"),
    ("el bloqueo de base de datos", "the database lock", "un candado que impide que dos transacciones modifiquen la misma fila a la vez", "a lock that prevents two transactions from modifying the same row at once"),
    ("la migración de esquema", "the schema migration", "un script versionado que cambia la estructura de la base de datos", "a versioned script that changes the structure of the database"),
    ("el registro de eventos", "event logging", "la práctica de escribir mensajes con marca de tiempo sobre lo que hace el sistema", "the practice of writing timestamped messages about what the system does"),
]
HERRAMIENTAS = ["PostgreSQL", "Redis", "nginx", "Docker", "pytest", "Kubernetes", "FastAPI", "SQLite", "Git", "Linux", "Celery", "Prometheus"]
PARAMETROS = ["max_connections", "timeout_seconds", "cache_size", "batch_size", "log_level", "retry_count", "pool_size", "num_workers"]
# aspecto -> (título ES, título EN, palabra clave ES, palabra clave EN, plantillas ES, plantillas EN)
ASPECTOS = {
    "intro": ("{n}: qué es y para qué sirve", "{n}: what it is and what it is for", "qué es", "what is",
        ["Se usa en {t} para simplificar el trabajo diario del equipo.", "Entender {c} es un requisito antes de pasar a producción.", "Su idea central es sencilla y aparece en casi todos los sistemas modernos.", "La documentación de {t} dedica un capítulo completo a {c}."],
        ["It is used in {t} to simplify the daily work of the team.", "Understanding {c} is a prerequisite before going to production.", "Its core idea is simple and appears in almost every modern system.", "The {t} documentation devotes a whole chapter to {c}."]),
    "config": ("{n}: cómo configurarlo", "{n}: how to configure it", "configurar", "configure",
        ["Para configurar {c} edita el archivo de ajustes y reinicia {t}.", "El parámetro {p} controla el comportamiento de {c} y su valor por defecto es {x}.", "Después de cambiar {p} a {x}, comprueba el resultado con un comando de diagnóstico.", "Guarda la configuración de {c} en un archivo versionado y no en la consola."],
        ["To configure {c}, edit the settings file and restart {t}.", "The {p} parameter controls the behavior of {c} and its default value is {x}.", "After changing {p} to {x}, check the result with a diagnostic command.", "Keep the configuration of {c} in a versioned file and not in the console."]),
    "errores": ("{n}: errores frecuentes", "{n}: common errors", "errores", "errors",
        ["Un error frecuente con {c} es olvidar el valor de {p} y dejarlo en {x}.", "Si aparece un fallo intermitente en {t}, revisa primero los registros de {c}.", "Muchos equipos descubren el problema de {c} solo cuando la carga se multiplica por {x}.", "El mensaje de error suele ser engañoso: la causa real está en {c}, no en {t}."],
        ["A common mistake with {c} is forgetting {p} and leaving it at {x}.", "If an intermittent failure shows up in {t}, check the logs of {c} first.", "Many teams only discover the problem with {c} when the load grows by {x} times.", "The error message is usually misleading: the real cause is in {c}, not in {t}."]),
    "rendimiento": ("{n}: rendimiento y optimización", "{n}: performance and optimization", "rendimiento", "performance",
        ["Para medir el rendimiento de {c} usa un perfilador y no tu intuición.", "Subir {p} a {x} mejora la latencia, pero consume más memoria en {t}.", "La optimización prematura de {c} suele complicar el código sin ganar tiempo.", "En una prueba con {x} usuarios, {c} redujo la latencia mediana a la mitad."],
        ["To measure the performance of {c}, use a profiler and not your intuition.", "Raising {p} to {x} improves latency but uses more memory in {t}.", "Premature optimization of {c} usually complicates the code without saving time.", "In a test with {x} users, {c} cut the median latency in half."]),
    "practicas": ("{n}: buenas prácticas", "{n}: best practices", "buenas prácticas", "best practices",
        ["Una buena práctica con {c} es documentar {p} y automatizar su despliegue en {t}.", "Revisa {c} en cada revisión de código y escribe pruebas para los casos límite.", "Mantén {c} simple: añade complejidad solo cuando los datos lo justifiquen.", "Define alertas sobre {c} antes del primer incidente, no después."],
        ["A good practice with {c} is to document {p} and automate its deployment in {t}.", "Review {c} in every code review and write tests for the edge cases.", "Keep {c} simple: add complexity only when the data justifies it.", "Define alerts on {c} before the first incident, not after."]),
    "ejemplo": ("{n}: ejemplo en Python", "{n}: Python example", "ejemplo", "example",
        ["En este ejemplo en Python usamos {t} para mostrar {c} paso a paso.", "El script define {p} = {x} y luego imprime el resultado de {c}.", "Ejecuta el ejemplo con Python 3.11 y compara la salida con la esperada.", "Puedes adaptar el código de {c} a tu propio proyecto cambiando solo {p}."],
        ["In this Python example we use {t} to show {c} step by step.", "The script defines {p} = {x} and then prints the result of {c}.", "Run the example with Python 3.11 and compare the output with the expected one.", "You can adapt the code for {c} to your own project by changing only {p}."]),
    "seguridad": ("{n}: consideraciones de seguridad", "{n}: security considerations", "seguridad", "security",
        ["Desde el punto de vista de la seguridad, {c} nunca debe exponerse sin autenticación.", "Rota las credenciales de {t} cada {x} días y no las guardes en el repositorio.", "Un ataque contra {c} suele aprovechar el valor de {p} mal configurado.", "Audita los accesos a {c} y conserva los registros durante {x} días."],
        ["From a security standpoint, {c} must never be exposed without authentication.", "Rotate the {t} credentials every {x} days and do not store them in the repository.", "An attack on {c} usually exploits a misconfigured {p} value.", "Audit access to {c} and keep the logs for {x} days."]),
}

In [ ]:
# ── Generador del corpus sintético (mismo esquema que el real) ────────────────
import random


def _es_fix(s: str) -> str:
    """Contracciones del español: 'a el' -> 'al', 'de el' -> 'del'."""
    return s.replace(" a el ", " al ").replace(" de el ", " del ").replace("De el ", "Del ").replace("A el ", "Al ")


def _sintetico_wiki(lang: str, n_docs: int, n_queries: int, seed: int = 42):
    """Docs técnicos sintéticos ES/EN: concepto x aspecto x herramienta. Devuelve (docs, queries, qrels)."""
    rng = random.Random(seed + (0 if lang == "es" else 1))
    li = 0 if lang == "es" else 1
    combos = [(c, a, t) for c in range(len(CONCEPTOS)) for a in ASPECTOS for t in range(len(HERRAMIENTAS))]
    rng.shuffle(combos)
    docs, titles = [], []
    for k, (ci, asp, ti) in enumerate(combos[:n_docs]):
        concepto, definicion = CONCEPTOS[ci][li], CONCEPTOS[ci][2 + li]
        tool = HERRAMIENTAS[ti]
        tit_t, plantillas = ASPECTOS[asp][li], ASPECTOS[asp][4 + li]
        nombre = concepto.split(" ", 1)[1] if concepto.split(" ", 1)[0] in ("el", "la", "los", "las", "the") else concepto
        slots = lambda: dict(c=concepto, t=tool, p=rng.choice(PARAMETROS), x=rng.choice([8, 16, 30, 64, 100, 256, 1024]))
        frases = [("{C} " + ("es " if li == 0 else "is ") + definicion + ".").format(C=concepto)]
        frases += [pl.format(**slots()) for pl in rng.sample(plantillas, len(plantillas))]
        frases.append(rng.choice(ASPECTOS["intro"][4 + li]).format(**slots()))
        for _ in range(2):  # ruido: frases de otro concepto (hace el retrieval no trivial)
            otro = rng.randrange(len(CONCEPTOS))
            otra = rng.choice(list(ASPECTOS.values()))[4 + li]
            s = slots(); s["c"] = CONCEPTOS[otro][li]
            frases.append(rng.choice(otra).format(**s))
        s = slots()
        frases.append(("Ejemplo: `{p} = {x}`." if li == 0 else "Example: `{p} = {x}`.").format(**s))
        text = " ".join(f[0].upper() + f[1:] for f in frases)
        docs.append({"id": f"{lang}-syn-{k:04d}", "lang": lang, "text": _es_fix(text) if lang == "es" else text})
        titles.append((nombre, ASPECTOS[asp][2 + li], tool, tit_t.format(n=nombre) + f" ({tool})"))
    queries, qrels = [], {}
    for j, k in enumerate(rng.sample(range(len(docs)), min(n_queries, len(docs)))):
        nombre, kw, tool, titulo = titles[k]
        q = titulo.replace(":", "").replace("(", "").replace(")", "") if j % 2 == 0 else f"{nombre} {kw} {tool}"
        qid = f"{lang}-q{j:04d}"
        queries.append({"qid": qid, "lang": lang, "text": q})
        qrels[qid] = {docs[k]["id"]: 1}
    return docs, queries, qrels

In [ ]:
# ── carga_corpus(): cadena oficial (HF) -> espejo (API de Wikipedia / stdlib) -> sintético ─────
import ast
import json
import os
import random
import urllib.parse
import urllib.request

SEED = 42


def _strip_docstring(func_src: str):
    """Devuelve (código sin docstring, primera frase del docstring) o None si no hay docstring útil."""
    try:
        fn = ast.parse(func_src).body[0]
        doc = ast.get_docstring(fn)
        first = fn.body[0]
    except Exception:
        return None
    if not doc or not (isinstance(first, ast.Expr) and isinstance(getattr(first, "value", None), ast.Constant)):
        return None
    lines = func_src.splitlines()
    code = "\n".join(lines[: first.lineno - 1] + lines[first.end_lineno :])
    frase = doc.strip().split("\n\n")[0].replace("\n", " ").strip()
    frase = frase.split(". ")[0].rstrip(".")
    return (code, frase) if len(frase.split()) >= 4 else None


def _code_stdlib(n_docs: int, n_queries: int, rng: random.Random):
    """Espejo OFFLINE con código real: funciones de la biblioteca estándar con su docstring como query."""
    import importlib, inspect, textwrap
    mods = ["argparse", "bisect", "calendar", "collections", "colorsys", "copy", "csv", "difflib", "fnmatch", "fractions",
            "functools", "getpass", "glob", "heapq", "html", "json", "operator", "os", "pathlib", "posixpath", "queue",
            "random", "re", "shlex", "shutil", "statistics", "string", "tempfile", "textwrap", "urllib.parse", "uuid", "zipfile"]
    cands = []
    for name in mods:
        try:
            src = inspect.getsource(importlib.import_module(name))
            tree = ast.parse(src)
        except Exception:
            continue
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.end_lineno - node.lineno >= 4:
                seg = textwrap.dedent("\n".join(src.splitlines()[node.lineno - 1 : node.end_lineno]))
                r = _strip_docstring(seg)
                if r and 200 <= len(r[0]) <= 2500:
                    cands.append((f"{name}.{node.name}", r[0], r[1]))
    rng.shuffle(cands)
    return cands[:n_docs]


def _code_hf(n_docs: int, rng: random.Random):
    """Oficial: CodeSearchNet (Python) en streaming. Prueba varios repos por cambios de formato en `datasets`."""
    from datasets import load_dataset
    errs = []
    for repo, kw in [("code-search-net/code_search_net", {"name": "python"}), ("code_search_net", {"name": "python"}),
                     ("claudios/code_search_net", {"name": "python"}), ("Nan-Do/code-search-net-python", {})]:
        try:
            ds = load_dataset(repo, split="train", streaming=True, **kw).shuffle(seed=SEED, buffer_size=2000)
            out = []
            for ex in ds:
                src = ex.get("func_code_string") or ex.get("whole_func_string") or ex.get("code") or ""
                r = _strip_docstring(src) if '"""' in src or "'''" in src else None
                doc = (ex.get("func_documentation_string") or ex.get("docstring") or "").strip().split("\n\n")[0]
                doc = doc.replace("\n", " ").split(". ")[0].rstrip(".")
                code = r[0] if r else src
                if 200 <= len(code) <= 2500 and len(doc.split()) >= 4:
                    out.append((ex.get("func_name") or f"csn.{len(out)}", code, doc))
                if len(out) >= n_docs:
                    return out
        except Exception as e:  # noqa: BLE001
            errs.append(f"{repo}: {type(e).__name__}")
    raise RuntimeError("CodeSearchNet no disponible -> " + "; ".join(errs))


def _wiki_hf(lang: str, n_docs: int, cap: int):
    """Oficial: wikimedia/wikipedia (snapshot 20231101) en streaming con un buffer pequeño."""
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", f"20231101.{lang}", split="train", streaming=True)
    ds = ds.shuffle(seed=SEED, buffer_size=2000)
    out = []
    for ex in ds:
        if len(ex["text"]) >= 600 and len(ex["title"]) >= 3:
            out.append((f"{lang}-{ex['id']}", ex["title"], ex["text"][:cap]))
        if len(out) >= n_docs:
            break
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out


def _wiki_api(lang: str, n_docs: int):
    """Espejo: API pública de Wikipedia (artículos aleatorios, solo la sección introductoria; ~20 por petición)."""
    out, seen, tries = [], set(), 0
    while len(out) < n_docs and tries < 80:
        tries += 1
        qs = urllib.parse.urlencode({"action": "query", "format": "json", "generator": "random", "grnnamespace": 0,
                                     "grnlimit": 20, "prop": "extracts", "exintro": 1, "explaintext": 1, "exlimit": 20})
        req = urllib.request.Request(f"https://{lang}.wikipedia.org/w/api.php?{qs}",
                                     headers={"User-Agent": "poliglota-course/1.0 (curso educativo)"})
        with urllib.request.urlopen(req, timeout=20) as r:
            pages = json.load(r).get("query", {}).get("pages", {})
        for p in pages.values():
            if p["pageid"] not in seen and len(p.get("extract", "")) >= 300:
                seen.add(p["pageid"])
                out.append((f"{lang}-{p['pageid']}", p["title"], p["extract"]))
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out[:n_docs]


def carga_corpus(fast_dev: bool = True) -> dict:
    """Corpus Políglota -> {"docs": [{id, lang, text}], "queries": [{qid, lang, text}], "qrels": {qid: {doc_id: 1}},
    "source": "real" | "synthetic"}. Cadena: oficial (HF) -> espejo (API Wikipedia; código de la stdlib) -> sintético."""
    rng = random.Random(SEED)
    n_wiki, n_code, q_wiki, q_code, cap = (150, 100, 40, 30, 1500) if fast_dev else (3000, 1500, 200, 150, 6000)
    docs, queries, qrels, source = [], [], {}, "real"
    wiki = {}
    for lang in ("es", "en"):
        for nombre, fn in (("HF wikimedia/wikipedia", lambda: _wiki_hf(lang, n_wiki, cap)),
                           ("API de Wikipedia (espejo)", lambda: _wiki_api(lang, min(n_wiki, 400)))):
            try:
                wiki[lang] = fn()
                print(f"[carga_corpus] Wikipedia {lang}: {len(wiki[lang])} artículos desde {nombre}")
                break
            except Exception as e:  # noqa: BLE001
                print(f"[carga_corpus] {nombre} ({lang}) falló: {type(e).__name__}: {str(e)[:80]}")
    if len(wiki) == 2:
        for lang, arts in wiki.items():
            docs += [{"id": i, "lang": lang, "text": t} for i, _, t in arts]
            for j, k in enumerate(rng.sample(range(len(arts)), min(q_wiki, len(arts)))):
                qid = f"{lang}-q{j:04d}"
                queries.append({"qid": qid, "lang": lang, "text": arts[k][1]})
                qrels[qid] = {arts[k][0]: 1}
    else:
        source = "synthetic"
        print("⚠️  [carga_corpus] SIN acceso a Wikipedia: uso texto ES/EN SINTÉTICO (mismo esquema). Los números NO son comparables con datos reales.")
        for lang in ("es", "en"):
            d, q, r = _sintetico_wiki(lang, n_wiki, q_wiki, SEED)
            docs += d; queries += q; qrels.update(r)
    try:
        code = _code_hf(n_code, rng)
        print(f"[carga_corpus] código: {len(code)} funciones desde CodeSearchNet (HF)")
    except Exception as e:  # noqa: BLE001
        print(f"[carga_corpus] CodeSearchNet falló ({str(e)[:90]}); uso funciones REALES de la stdlib de Python (espejo offline)")
        code = _code_stdlib(n_code, q_code, rng)
    for k, (name, src, doc) in enumerate(code):
        docs.append({"id": f"code-{k:04d}", "lang": "code", "text": src})
    for j, k in enumerate(rng.sample(range(len(code)), min(q_code, len(code)))):
        qid = f"code-q{j:04d}"
        queries.append({"qid": qid, "lang": "code", "text": code[k][2]})
        qrels[qid] = {f"code-{k:04d}": 1}
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": source}

In [ ]:
# ── Métricas de retrieval canónicas de Políglota (módulo 00) ──────────────────
import math

import numpy as np


def _dedup(ranked_ids):
    """Quita duplicados conservando la primera aparición (un doc repetido NO puntúa dos veces)."""
    seen, out = set(), []
    for d in ranked_ids:
        if d not in seen:
            seen.add(d)
            out.append(d)
    return out


def dcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """DCG con ganancia lineal (como trec_eval / pytrec_eval, que usa BEIR/MTEB) y descuento 1/log2(rango+1)."""
    ranked = _dedup(ranked_ids)[:k]
    return sum(rels.get(d, 0) / math.log2(i + 2) for i, d in enumerate(ranked) if rels.get(d, 0) > 0)


def ndcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """nDCG@k. rels: doc_id -> relevancia (>0 relevante). Sin relevantes -> 0.0 (evaluate() las excluye)."""
    ideal = sorted((r for r in rels.values() if r > 0), reverse=True)[:k]
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(ideal))
    return 0.0 if idcg == 0 else dcg_at_k(ranked_ids, rels, k) / idcg


def recall_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """Fracción de los documentos relevantes que aparecen en el top-k."""
    relevant = {d for d, r in rels.items() if r > 0}
    if not relevant:
        return 0.0
    return len(relevant & set(_dedup(ranked_ids)[:k])) / len(relevant)


def mrr_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """1/rango del primer relevante dentro del top-k (0 si no hay ninguno)."""
    for i, d in enumerate(_dedup(ranked_ids)[:k]):
        if rels.get(d, 0) > 0:
            return 1.0 / (i + 1)
    return 0.0


def rank_from_scores(scores, doc_ids, k: int | None = None):
    """Ordena por score desc. Empates -> orden original del corpus (determinista). Los empates son una fuente
    silenciosa de varianza: con scores iguales, el orden de entrada decidiría el resultado."""
    scores = np.asarray(scores, dtype=float)
    order = np.lexsort((np.arange(len(scores)), -scores))
    order = order[:k] if k else order
    return [doc_ids[i] for i in order]


def evaluate(run: dict, qrels: dict, k: int = 10) -> dict:
    """run: {qid: [doc_id ordenados]}. Promedia solo sobre queries con algún relevante (convención trec_eval)."""
    n, tot = 0, {"ndcg": 0.0, "recall": 0.0, "mrr": 0.0}
    for qid, rels in qrels.items():
        if not any(r > 0 for r in rels.values()):
            continue
        ranked = run.get(qid, [])
        tot["ndcg"] += ndcg_at_k(ranked, rels, k)
        tot["recall"] += recall_at_k(ranked, rels, k)
        tot["mrr"] += mrr_at_k(ranked, rels, k)
        n += 1
    out = {m: (v / n if n else 0.0) for m, v in tot.items()}
    out.update(n_queries=n, n_skipped=len(qrels) - n)
    return out

In [ ]:
# ── Tokenización simple + BM25 propio (módulo 00) ─────────────────────────────
import math
import re
import unicodedata
from collections import Counter, defaultdict

import numpy as np

_WORD_RE = re.compile(r"\w+", re.UNICODE)


def fold_accents(text: str) -> str:
    """'canción' -> 'cancion' (NFD y descartar marcas combinantes Mn)."""
    nfd = unicodedata.normalize("NFD", text)
    return unicodedata.normalize("NFC", "".join(c for c in nfd if unicodedata.category(c) != "Mn"))


def tokenize_simple(text: str, accents: str = "keep") -> list[str]:
    """Minúsculas + palabras \\w+. accents='fold' quita tildes. (Esto NO es el 'token' de un LLM: es un token léxico.)"""
    text = unicodedata.normalize("NFC", text).lower()
    if accents == "fold":
        text = fold_accents(text)
    return _WORD_RE.findall(text)


class BM25:
    """BM25 con índice invertido. idf='lucene': ln(1+(N-n+.5)/(n+.5)) (siempre >= 0).
    idf='okapi': ln((N-n+.5)/(n+.5)) con suelo epsilon*idf_medio (como rank_bm25.BM25Okapi)."""

    def __init__(self, k1: float = 1.2, b: float = 0.75, idf: str = "lucene", epsilon: float = 0.25):
        self.k1, self.b, self.idf_kind, self.epsilon = k1, b, idf, epsilon

    def fit(self, docs_tokens: list[list[str]], doc_ids: list[str]) -> "BM25":
        self.doc_ids = list(doc_ids)
        self.N = len(docs_tokens)
        self.dl = np.array([len(t) for t in docs_tokens], dtype=float)
        self.avgdl = float(self.dl.mean()) if self.N else 0.0
        self.postings = defaultdict(list)  # término -> [(índice_doc, tf)]
        for i, toks in enumerate(docs_tokens):
            for term, tf in Counter(toks).items():
                self.postings[term].append((i, tf))
        raw = {t: len(p) for t, p in self.postings.items()}
        if self.idf_kind == "lucene":
            self.idf = {t: math.log(1 + (self.N - n + 0.5) / (n + 0.5)) for t, n in raw.items()}
        else:
            self.idf = {t: math.log(self.N - n + 0.5) - math.log(n + 0.5) for t, n in raw.items()}
            floor = self.epsilon * sum(self.idf.values()) / len(self.idf)
            self.idf = {t: (v if v >= 0 else floor) for t, v in self.idf.items()}
        return self

    def scores(self, q_tokens: list[str]) -> np.ndarray:
        s = np.zeros(self.N)
        for term in q_tokens:  # los términos repetidos de la query suman varias veces (igual que rank_bm25)
            for i, tf in self.postings.get(term, ()):
                norm = self.k1 * (1 - self.b + self.b * self.dl[i] / self.avgdl)
                s[i] += self.idf[term] * tf * (self.k1 + 1) / (tf + norm)
        return s

    def search(self, q_tokens: list[str], k: int = 10) -> list[str]:
        s = self.scores(q_tokens)
        order = np.lexsort((np.arange(self.N), -s))[:k]  # empates -> orden del corpus
        return [self.doc_ids[i] for i in order]

In [ ]:
corpus = carga_corpus(FAST_DEV_RUN)
docs, queries, qrels = corpus["docs"], corpus["queries"], corpus["qrels"]
doc_ids = [d["id"] for d in docs]
doc_txt = [d["text"] for d in docs]
q_txt = [q["text"] for q in queries]
print("fuente:", corpus["source"], "| docs:", dict(Counter(d["lang"] for d in docs)), "| queries:", dict(Counter(q["lang"] for q in queries)))


def evalua(S, k=10):
    """S: (n_queries, n_docs) similitudes -> métricas canónicas @k sobre todas las queries."""
    run = {q["qid"]: rank_from_scores(S[i], doc_ids, 100) for i, q in enumerate(queries)}
    return evaluate(run, qrels, k)

## 1. Intuición: de un vector por token a un vector por frase

En un Transformer, el vector de un token en la capa $\ell$ es $\mathbf h^{(\ell)}_i = f_\ell(\mathbf x_1,\dots,\mathbf x_T)_i$: depende de **toda** la frase. «banco» en *«me senté en el banco del parque»* y en *«el banco subió los tipos»* entra con la misma fila de $E$ (capa 0) y sale con vectores distintos en las capas altas.
**BERT** (Devlin et al., 2019) es un encoder bidireccional preentrenado con *masked language modeling*; su token especial `[CLS]` se pensó como resumen para clasificación.

Para buscar documentos necesitamos un **único** vector por texto: $\mathbf s = \text{pool}(\mathbf h_1,\dots,\mathbf h_T)$. La pregunta del módulo es **qué pooling y qué post-procesado** hacen que $\cos(\mathbf s_q,\mathbf s_d)$ refleje relevancia.

> 💡 **Analogía con recsys.** En un *two-tower* (`recsys-course/08_deep_retrieval_ann`) cada torre convierte un usuario/ítem con muchas *features* en un vector y el score es un producto escalar. Un **bi-encoder** de texto es lo mismo: la torre es un Transformer + pooling, y el «ítem» es un documento. Todo lo que sabes de *two-tower* (indexabilidad del producto escalar, sesgo por normas, ANN) aplica aquí.

> ### 🎨 GEMINI-IMAGEN · ID: `M10-IMG-01` · ⚠️ REEMPLAZAR ESTA CELDA
> **Estático vs contextual: la palabra «banco»**
>
> **Qué muestra:** Contrasta un embedding estático (una fila de E, igual en cualquier frase) con los embeddings contextuales de la última capa, que separan los dos sentidos de «banco».
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Comparación de embedding estático (un punto para «banco») y contextual (dos puntos según el sentido).
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, 16:9, sin decoración. Paleta: azul #1F6FEB (sentido «asiento»), naranja #F28E2B (sentido «entidad financiera»), gris #6B7280 (elementos neutros), verde #2CA02C (flechas de contexto). Tipografía sans-serif.
> Mitad izquierda, título «Embedding estático (fila de E)»: dos frases en cajas grises: «Me senté en el banco del parque» y «El banco subió los tipos de interés». De la palabra «banco» de ambas frases salen flechas grises hacia UN ÚNICO punto gris en un plano 2D rotulado «E[banco]» con la nota «mismo vector en las dos frases».
> Mitad derecha, título «Embedding contextual (capa final del Transformer)»: las mismas dos frases; flechas verdes desde las palabras vecinas («parque», «senté» / «tipos», «interés») hacia «banco» indicando atención. En un plano 2D, un punto azul rotulado «banco (asiento)» cerca de puntos azules pequeños «silla», «parque», y un punto naranja «banco (entidad)» cerca de puntos naranjas «crédito», «tipos». Línea discontinua entre ambos puntos con el texto «coseno bajo».
> Pie común: «Capa 0 = estático; capas altas = cada vez más dependiente del contexto».
> Evita: texto ilegible, flechas ambiguas, números inventados, iconos decorativos.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M10-IMG-01.png` en la carpeta del módulo y reemplaza esta celda por: `![Comparación de embedding estático (un punto para «banco») y contextual (dos puntos según el sentido).](assets/M10-IMG-01.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> ESTÁTICO (E):   "banco" (parque) ─┐
>                 "banco" (tipos)  ─┴─► • E[banco]   (mismo vector)
> CONTEXTUAL:     "banco" (parque) ─► • cerca de {silla, parque}
>                 "banco" (tipos)  ─► • cerca de {crédito, tipos}   (coseno bajo)
> ```

> 👀 **Qué debes observar.** La diferencia no es «más dimensiones», sino que el vector **depende de los vecinos** vía atención. Por eso el embedding de una frase puede capturar el sentido, cosa imposible promediando filas de $E$.

## 2. Pooling desde cero
Dados los estados $H\in\mathbb R^{B\times T\times d}$ de un lote y la máscara $M\in\{0,1\}^{B\times T}$ (1 = token real, 0 = *padding*):

| Pooling | Fórmula | Típico en |
|---|---|---|
| CLS | $\mathbf s=\mathbf h_1$ | BERT original, DPR |
| media | $\mathbf s=\dfrac{\sum_i M_i\mathbf h_i}{\sum_i M_i}$ | SBERT (por defecto), E5 |
| máximo | $s_k=\max_{i:M_i=1} h_{ik}$ | variante de SBERT |
| último token | $\mathbf s=\mathbf h_{i^*}$, $i^*$ = último token real | LLMs decoder (SGPT, e5-mistral; módulo 12) |

La máscara **no es opcional**: sin ella, el padding cambia el embedding de un texto según con quién compartió lote.

✍️ **Ejemplo a mano ($d = 2$, $T = 4$).** Estados $\mathbf h_1 = (1, 0)$ ([CLS]), $\mathbf h_2 = (0, 2)$, $\mathbf h_3 = (2, 2)$ y una posición de *padding* $\mathbf h_4 = (9, 9)$, máscara $M = (1, 1, 1, 0)$. CLS $= (1, 0)$; media con máscara $= \tfrac13(3, 4) = (1, 1.33)$; máximo con máscara $= (2, 2)$; último token real $= \mathbf h_3 = (2, 2)$. **Sin** máscara, la media sería $\tfrac14(12, 13) = (3, 3.25)$: el padding domina. Compruébalo con la celda siguiente antes de leer su salida. Analogía IR: la media es una «bolsa de palabras» densa (cada token vota igual, como el TF de BM25 sin saturar), y el CLS es un resumen que el modelo aprende a escribir.

In [ ]:
# ── Pooling desde cero: de (B, T, d) estados por token + máscara (B, T) a (B, d) un vector por texto ──
def media_mascara(h, mask):
    m = mask.unsqueeze(-1).to(h.dtype)
    return (h * m).sum(1) / m.sum(1).clamp(min=1e-9)


def max_mascara(h, mask):
    return h.masked_fill(mask.unsqueeze(-1) == 0, torch.finfo(h.dtype).min).max(1).values


def cls_pool(h, mask):
    return h[:, 0]


def ultimo_token(h, mask):
    """Último token REAL de cada fila (sirve con padding a la derecha o a la izquierda)."""
    T = mask.shape[1]
    idx = T - 1 - mask.flip(1).float().argmax(1)          # posición del último 1 de cada fila
    return h[torch.arange(h.shape[0]), idx]


POOLS = {"cls": cls_pool, "mean": media_mascara, "max": max_mascara, "last": ultimo_token}

In [ ]:
# El padding importa: la media SIN máscara depende del lote; con máscara, no.
torch.manual_seed(SEED)
h = torch.randn(2, 6, 4)
mask = torch.tensor([[1, 1, 1, 1, 1, 1], [1, 1, 1, 0, 0, 0]])         # el 2.º texto tiene 3 tokens + 3 de padding
solo = media_mascara(h[1:, :3], torch.ones(1, 3, dtype=torch.long))    # el mismo texto codificado solo, sin padding
print("media CON máscara == texto solo:", torch.allclose(media_mascara(h, mask)[1], solo[0]))
print("media SIN máscara == texto solo:", torch.allclose(h.mean(1)[1], solo[0]))
print("último token real por fila:", (mask.shape[1] - 1 - mask.flip(1).float().argmax(1)).tolist(), "(padding a la derecha)")
mask_izq = torch.tensor([[1, 1, 1, 1, 1, 1], [0, 0, 0, 1, 1, 1]])
print("último token real con padding a la izquierda:", (mask_izq.shape[1] - 1 - mask_izq.flip(1).float().argmax(1)).tolist())

> 👀 **Qué debes observar.** `True` con máscara y `False` sin ella: un bug de pooling sin máscara hace que **el mismo documento tenga embeddings distintos** según el lote (y el índice no es reproducible). `ultimo_token` funciona con padding a la izquierda o a la derecha; `h[:, -1]` solo con padding a la izquierda (error clásico con LLMs-embedders, módulo 12).

## 3. Encoders: reales y *fallback*
La celda carga tres encoders multilingües reales del hub (si hay acceso):
- **BERT multilingüe crudo** (`distilbert-base-multilingual-cased`): preentrenado solo con MLM, **sin** ajustar para similitud.
- **SBERT multilingüe** (`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`): bi-encoder ajustado para similitud y multilingüe por destilación (Reimers & Gurevych, 2020).
- **E5 multilingüe small** (`intfloat/multilingual-e5-small`): entrenado contrastivamente con prefijos `query:` / `passage:` (Wang et al., 2022).

Sin acceso al hub se construyen dos encoders diminutos **locales** con la misma API (fallback documentado): un mini-BERT solo-MLM («crudo») y su versión con 200 pasos de ajuste contrastivo con pares débiles («mini-SBERT»). Los números con el fallback **no** son comparables con los de los modelos reales; los mecanismos sí.

In [ ]:
# ── Fallback documentado: si no hay acceso al hub, construimos dos encoders diminutos LOCALES con la API de HF ──
#    mini-BERT "crudo" (solo MLM, como un BERT sin ajustar) y mini-SBERT (el mismo + 200 pasos contrastivos con
#    pares débiles frase-1 ↔ resto-del-documento; el módulo 11 explica este entrenamiento paso a paso).
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers
from transformers import BertConfig, BertForMaskedLM, BertModel, BertTokenizerFast, DataCollatorForLanguageModeling


def vocab_wordpiece_determinista(textos, vocab_size=2000):
    """Vocabulario con convención WordPiece ("##" = continuación de palabra), REPRODUCIBLE entre ejecuciones.
    El WordPiece trainer de `tokenizers` rompe empates según el orden de un HashMap de Rust, que cambia en cada proceso:
    el vocabulario (y con él todo lo que se entrena encima) variaba entre ejecuciones. Unigram sí es determinista."""
    uni = Tokenizer(models.Unigram())
    uni.normalizer = normalizers.BertNormalizer(lowercase=True, strip_accents=False)
    uni.pre_tokenizer = pre_tokenizers.Sequence([pre_tokenizers.BertPreTokenizer(), pre_tokenizers.Metaspace()])
    uni.train_from_iterator(textos, trainers.UnigramTrainer(vocab_size=vocab_size, unk_token="[UNK]",
                                                            special_tokens=["[UNK]"], show_progress=False))
    chars = sorted({c for t in textos for c in uni.normalizer.normalize_str(t) if not c.isspace()})
    piezas = [p for p, _ in sorted(uni.get_vocab().items(), key=lambda x: x[1]) if p not in ("[UNK]", "▁")]
    piezas = [p[1:] if p.startswith("▁") else "##" + p for p in piezas]       # "▁x" inicia palabra; "x" → "##x"
    vocab = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"] + chars + ["##" + c for c in chars] + piezas
    return list(dict.fromkeys(vocab))


def construye_fallback(textos, out="mini_encoders", steps_mlm=200, steps_cl=200, seed=SEED):
    os.makedirs(f"{out}/raw", exist_ok=True)
    with open(f"{out}/raw/vocab.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(vocab_wordpiece_determinista(textos)) + "\n")
    tk = BertTokenizerFast(f"{out}/raw/vocab.txt", do_lower_case=True, strip_accents=False)
    torch.manual_seed(seed); rng = random.Random(seed)
    mlm = BertForMaskedLM(BertConfig(vocab_size=len(tk), hidden_size=128, num_hidden_layers=2, num_attention_heads=4,
                                     intermediate_size=256, max_position_embeddings=128))
    coll = DataCollatorForLanguageModeling(tk, mlm_probability=0.15)
    feats = [{"input_ids": x} for x in tk(textos, truncation=True, max_length=64)["input_ids"]]
    opt = torch.optim.AdamW(mlm.parameters(), lr=1e-3)
    for _ in range(steps_mlm):
        loss = mlm(**coll([feats[rng.randrange(len(feats))] for _ in range(32)])).loss
        opt.zero_grad(); loss.backward(); opt.step()
    mlm.bert.save_pretrained(f"{out}/raw"); tk.save_pretrained(f"{out}/raw")
    # pares débiles (sin etiquetas): primera frase <-> resto del documento
    pares = []
    for t in textos:
        fr = re.split(r"(?<=[.!?])\s+", t.strip())
        if len(fr) >= 3:
            pares.append((fr[0], " ".join(fr[1:])))
    enc = copy.deepcopy(mlm.bert); opt = torch.optim.AdamW(enc.parameters(), lr=5e-4)
    def emb(b):  # media con máscara (autocontenida para no depender de otras celdas)
        m = b["attention_mask"].unsqueeze(-1).float()
        return F.normalize((enc(**b).last_hidden_state * m).sum(1) / m.sum(1), dim=-1)
    for _ in range(steps_cl):
        lote = rng.sample(pares, 32)
        a = tk([p[0] for p in lote], padding=True, truncation=True, max_length=64, return_tensors="pt")
        b = tk([p[1] for p in lote], padding=True, truncation=True, max_length=64, return_tensors="pt")
        loss = F.cross_entropy(emb(a) @ emb(b).T / 0.05, torch.arange(len(lote)))
        opt.zero_grad(); loss.backward(); opt.step()
    enc.save_pretrained(f"{out}/sbert"); tk.save_pretrained(f"{out}/sbert")
    return f"{out}/raw", f"{out}/sbert"

In [ ]:
from transformers import AutoModel, AutoTokenizer


class Encoder:
    """Envuelve cualquier encoder de HF: estados por token de cualquier capa + pooling elegido. prefijos = (query, doc)."""

    def __init__(self, nombre, ruta, prefijos=("", ""), max_len=128):
        self.nombre, self.ruta, self.prefijos, self.max_len = nombre, ruta, prefijos, max_len
        self.tok = AutoTokenizer.from_pretrained(ruta)
        self.model = AutoModel.from_pretrained(ruta).to(device).eval()

    @torch.no_grad()
    def estados(self, textos, capa=-1, bs=32):
        """Lista de (h (b,T,d) en CPU, mask (b,T)) por lotes."""
        out = []
        for i in range(0, len(textos), bs):
            b = self.tok(textos[i:i + bs], padding=True, truncation=True, max_length=self.max_len, return_tensors="pt").to(device)
            hs = self.model(**b, output_hidden_states=True).hidden_states
            out.append((hs[capa].float().cpu(), b["attention_mask"].cpu()))
        return out

    def encode(self, textos, pooling="mean", capa=-1, tipo="doc"):
        pre = self.prefijos[0] if tipo == "query" else self.prefijos[1]
        return torch.cat([POOLS[pooling](h, m) for h, m in self.estados([pre + t for t in textos], capa)]).numpy()


def carga_encoders(textos):
    """Modelos reales (si hay hub) o los mini-encoders de fallback. Devuelve {nombre: Encoder}."""
    reales = [("BERT multilingüe crudo", "distilbert-base-multilingual-cased", ("", "")),
              ("SBERT multilingüe", "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", ("", "")),
              ("E5 multilingüe small", "intfloat/multilingual-e5-small", ("query: ", "passage: "))]
    encs = {}
    for nombre, ruta, pre in reales:
        try:
            encs[nombre] = Encoder(nombre, ruta, pre)
        except Exception as e:
            print(f"   (sin {ruta}: {type(e).__name__})")
    if not encs:
        print("⚠️  Sin acceso al hub: construyo los mini-encoders de fallback (~1-2 min en CPU)")
        raw, sb = construye_fallback(textos)
        encs = {"mini-BERT crudo (fallback)": Encoder("mini-BERT crudo (fallback)", raw),
                "mini-SBERT (fallback)": Encoder("mini-SBERT (fallback)", sb)}
    return encs

Utilidades de similitud y post-procesado (`coseno`, `anisotropia`, whitening). Las explicamos en la sección 6; aquí solo las definimos.

In [ ]:
# ── Post-procesado desde cero: centrado, whitening (Su et al., 2021) y "all-but-the-top" (Mu & Viswanath, 2018) ──
def ajusta_whitening(X, k=None, eps=1e-6):
    """Devuelve (mu, W) tales que (x - mu) @ W tiene covarianza identidad en las k primeras direcciones."""
    mu = X.mean(0, keepdims=True)
    cov = np.cov((X - mu).T)
    U, S, _ = np.linalg.svd(cov)
    k = k or X.shape[1]
    return mu, U[:, :k] / np.sqrt(S[:k] + eps)


def quita_top(X, mu, n_top=1):
    """All-but-the-top: centra y elimina las n_top direcciones principales."""
    Xc = X - mu
    U, _, _ = np.linalg.svd(np.cov(Xc.T))
    P = U[:, :n_top]
    return Xc - Xc @ P @ P.T


def coseno(Q, D):
    Qn = Q / np.linalg.norm(Q, axis=1, keepdims=True).clip(1e-12)
    Dn = D / np.linalg.norm(D, axis=1, keepdims=True).clip(1e-12)
    return Qn @ Dn.T


def anisotropia(X, n=3000, seed=SEED):
    r = np.random.RandomState(seed)
    i, j = r.randint(0, len(X), n), r.randint(0, len(X), n)
    Xn = X / np.linalg.norm(X, axis=1, keepdims=True).clip(1e-12)
    keep = i != j
    return float((Xn[i[keep]] * Xn[j[keep]]).sum(1).mean())

In [ ]:
t0 = time.time()
encs = carga_encoders(doc_txt)
print("encoders:", list(encs), f"({time.time() - t0:.0f}s)")

## 4. ¿Cuán contextuales son las representaciones? (por capa)
Ethayarajh (2019) define la **self-similarity** de una palabra en la capa $\ell$ como el coseno medio entre sus representaciones en contextos distintos, y la compara con la **anisotropía** de esa capa (coseno medio entre tokens aleatorios). Si ambas son altas, la palabra *parece* estable solo porque *todo* es parecido; la medida útil es la diferencia.

In [ ]:
def contextualidad(enc, textos, max_docs=60):
    """Por capa: (self-similarity del token más frecuente de ≥ 5 letras, anisotropía entre tokens aleatorios)."""
    ids_txt = enc.tok(textos, truncation=True, max_length=enc.max_len)["input_ids"]
    cnt = Counter(i for ids in ids_txt for i in set(ids))
    vocab_inv = {v: k for k, v in enc.tok.get_vocab().items()}
    cand = [i for i, n in cnt.most_common() if re.fullmatch(r"[^\W\d_]{5,}", vocab_inv[i].lstrip("▁Ġ#"))]
    wid = cand[0]
    con = [t for t, ids in zip(textos, ids_txt) if wid in ids][:max_docs]
    b = enc.tok(con, padding=True, truncation=True, max_length=enc.max_len, return_tensors="pt").to(device)
    with torch.no_grad():
        hs = enc.model(**b, output_hidden_states=True).hidden_states
    pos = (b["input_ids"] == wid) & (b["attention_mask"] == 1)
    res_ = []
    for h in hs:
        h = h.float().cpu()
        w = F.normalize(h[pos.cpu()], dim=-1)
        ss = (w @ w.T)[~torch.eye(len(w), dtype=bool)].mean().item()
        toks = h[b["attention_mask"].cpu() == 1]
        res_.append((ss, anisotropia(toks.numpy())))
    return vocab_inv[wid], np.array(res_)


fig, ax = plt.subplots(1, len(encs), figsize=(5.2 * len(encs), 3.4), squeeze=False)
for a, (n, e) in zip(ax[0], encs.items()):
    palabra, r_ = contextualidad(e, doc_txt)
    capas = np.arange(len(r_))
    a.plot(capas, r_[:, 0], "o-", color=PAL["azul"], label=f"self-similarity «{palabra}»")
    a.plot(capas, r_[:, 1], "s--", color=PAL["naranja"], label="anisotropía (tokens aleatorios)")
    a.plot(capas, r_[:, 0] - r_[:, 1], "^-", color=PAL["verde"], label="self-sim − anisotropía")
    a.set(xlabel="capa (0 = embeddings de entrada)", ylabel="coseno medio", title=n); a.legend(fontsize=7)
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** En la capa 0 la *self-similarity* es 1 (es la misma fila de $E$, salvo la suma de posiciones) y cae al subir: las capas altas son **más contextuales**. La anisotropía (naranja) no es cero en ninguna capa: Ethayarajh (2019) la midió alta en las capas superiores de BERT y GPT-2, y Godey et al. (2024) argumentan que es en buena parte inherente a la auto-atención. La curva verde es la contextualidad «real».

## 5. Pooling × modelo en Políglota
Codificamos documentos y consultas con cada pooling y medimos nDCG@10 con coseno. Recuerda la referencia: **BM25** del módulo 00 en este corpus.

> ### 🎨 GEMINI-IMAGEN · ID: `M10-IMG-02` · ⚠️ REEMPLAZAR ESTA CELDA
> **Cuatro formas de resumir una secuencia (pooling)**
>
> **Qué muestra:** Muestra sobre la misma matriz de estados ocultos qué posiciones usa cada pooling (CLS, media, máximo, último token) y el papel de la máscara de padding.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Cuatro paneles que muestran qué estados ocultos usa cada pooling y que el padding se excluye con la máscara.
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, 16:9. Paleta: azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280; tipografía sans-serif.
> Arriba: una frase tokenizada en 8 fichas: «[CLS]», «cómo», «configurar», «la», «caché», «[SEP]», «[PAD]», «[PAD]». Las dos fichas [PAD] en gris claro con la etiqueta «máscara = 0».
> Debajo, una cuadrícula de 8 columnas × 6 filas (estados ocultos h₁…h₈ de dimensión d), con las columnas [PAD] sombreadas en gris y tachadas.
> Debajo, cuatro paneles en fila, cada uno con una flecha desde la cuadrícula a un vector de 6 casillas:
> 1) «CLS»: solo la columna 1 resaltada en azul.
> 2) «media (con máscara)»: columnas 1–6 resaltadas en verde y la fórmula «Σ mᵢ hᵢ / Σ mᵢ».
> 3) «máximo»: casillas sueltas resaltadas en naranja en distintas columnas (el máximo por dimensión), nunca en [PAD].
> 4) «último token real»: solo la columna 6 ([SEP]) resaltada en azul oscuro con la nota «no la columna 8».
> Pie: «El pooling debe coincidir con el que se usó al entrenar el modelo».
> Evita: texto ilegible, flechas ambiguas, números inventados, decoración.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M10-IMG-02.png` en la carpeta del módulo y reemplaza esta celda por: `![Cuatro paneles que muestran qué estados ocultos usa cada pooling y que el padding se excluye con la máscara.](assets/M10-IMG-02.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> tokens: [CLS] cómo configurar la caché [SEP] [PAD] [PAD]
> máscara:  1    1      1       1    1     1     0     0
> CLS   -> h1           media -> Σ m_i h_i / Σ m_i (h1..h6)
> max   -> max por dim (sin PAD)   último -> h6 (último real, no h8)
> ```

> 👀 **Qué debes observar.** Cada pooling es una decisión de **entrenamiento**, no solo de inferencia: un modelo entrenado con media espera media. La máscara aparece en tres de los cuatro paneles.

In [ ]:
bm25 = BM25().fit([tokenize_simple(t) for t in doc_txt], doc_ids)          # BM25 canónico del módulo 00
S_bm25 = np.array([bm25.scores(tokenize_simple(q)) for q in q_txt])
ndcg_bm25 = evalua(S_bm25)["ndcg"]

tabla = {}
for n, e in encs.items():
    for p in POOLS:
        D, Q = e.encode(doc_txt, p, tipo="doc"), e.encode(q_txt, p, tipo="query")
        tabla[(n, p)] = evalua(coseno(Q, D))["ndcg"]
fig, ax = plt.subplots(figsize=(8.5, 3.6))
x = np.arange(len(POOLS)); w = 0.8 / len(encs)
for k, (n, col) in enumerate(zip(encs, [PAL["gris"], PAL["azul"], PAL["verde"]])):
    vals = [tabla[(n, p)] for p in POOLS]
    ax.bar(x + k * w - 0.4 + w / 2, vals, w, color=col, label=n)
ax.axhline(ndcg_bm25, ls="--", color=PAL["naranja"], label=f"BM25 (módulo 00) = {ndcg_bm25:.3f}")
ax.set_xticks(x, list(POOLS)); ax.set(ylabel="nDCG@10", title="Pooling × encoder en Políglota (coseno)"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()
for (n, p), v in tabla.items():
    print(f"{n:30s} {p:5s} nDCG@10={v:.3f}")

> 👀 **Qué debes observar.** (1) El encoder **crudo** (solo MLM) es el peor con cualquier pooling: preentrenar para predecir palabras no basta para que el coseno mida relevancia. SBERT lo documentó: con BERT sin ajustar, el promedio de STS fue 54.81 (media) y 29.19 (CLS), por debajo de promediar GloVe (61.32) (Reimers & Gurevych, 2019, tabla 1).
> (2) El pooling que mejor funciona es, casi siempre, **el que se usó al entrenar** (media para SBERT/E5). (3) En este corpus de «ítem conocido», BM25 es un rival durísimo: las consultas comparten palabras exactas con su documento. En el fallback los mini-encoders quedan muy por debajo; con los modelos reales, compara tú mismo.

In [ ]:
# Librería de industria: sentence-transformers con módulos Transformer + Pooling. ¿Da lo mismo que nuestro pooling?
try:
    from sentence_transformers import SentenceTransformer
    from sentence_transformers.models import Pooling, Transformer
    e = next(iter(encs.values()))
    tr = Transformer(e.ruta, max_seq_length=e.max_len)
    st = SentenceTransformer(modules=[tr, Pooling(tr.get_word_embedding_dimension(), pooling_mode="mean")], device=device)
    ref = st.encode(doc_txt[:16], convert_to_numpy=True, batch_size=8)
    mio = e.encode(doc_txt[:16], "mean")
    print(f"{e.nombre}: max |ST − nuestro mean pooling| = {np.abs(ref - mio).max():.2e}")
except Exception as ex:
    print("sentence-transformers no disponible:", type(ex).__name__, ex)

> 👀 **Qué debes observar.** Diferencia del orden de $10^{-6}$ o menor: `sentence-transformers` hace exactamente la media con máscara. En producción usarás la librería (gestiona lotes, *truncation*, normalización y configuración guardada con el modelo); implementarlo una vez te permite **auditar** que el pooling del modelo coincide con el de tu índice.

## 6. Anisotropía, dimensiones dominantes y cómo corregirlas sin entrenar

**Anisotropía.** Si todos los embeddings comparten una dirección grande, $\cos(\mathbf s_a,\mathbf s_b)$ es alto para *cualquier* par y el ranking lo deciden diferencias pequeñas. Además, unas pocas **dimensiones dominantes** (*rogue dimensions*, Timkey & van Schijndel, 2021) pueden concentrar casi todo el coseno.
La contribución de la dimensión $k$ al coseno medio entre pares es $\;c_k=\mathbb E\big[\tfrac{u_k v_k}{\|\mathbf u\|\|\mathbf v\|}\big]$, con $\sum_k c_k$ = coseno medio.

In [ ]:
def contribucion_dims(X, n=3000, seed=SEED):
    r = np.random.RandomState(seed)
    i, j = r.randint(0, len(X), n), r.randint(0, len(X), n)
    Xn = X / np.linalg.norm(X, axis=1, keepdims=True).clip(1e-12)
    return (Xn[i] * Xn[j]).mean(0)                        # vector (d,) que suma el coseno medio


fig, ax = plt.subplots(1, len(encs), figsize=(5.2 * len(encs), 3.2), squeeze=False)
EMB = {}
for a, (n, e) in zip(ax[0], encs.items()):
    D, Q = e.encode(doc_txt, "mean"), e.encode(q_txt, "mean", tipo="query")
    EMB[n] = (D, Q)
    c = contribucion_dims(D)
    top = np.argsort(-np.abs(c))[:3]
    a.bar(np.arange(len(c)), c, color=PAL["gris"])
    a.bar(top, c[top], color=PAL["naranja"])
    a.set(xlabel="dimensión", ylabel="contribución al coseno medio",
          title=f"{n}\ncos medio={c.sum():.2f} · top-3 dims={c[top].sum() / c.sum():.0%}")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** La suma de barras es el coseno medio entre documentos aleatorios (la anisotropía). Si 3 dimensiones naranjas explican una fracción grande, el coseno está dominado por ellas: es lo que Timkey & van Schijndel (2021) llaman *rogue dimensions*, y por eso el centrado o la estandarización por dimensión pueden cambiar mucho el ranking.

### 📐 Whitening, paso a paso (Su et al., 2021)
Queremos una transformación lineal $\tilde{\mathbf x}=(\mathbf x-\boldsymbol\mu)W$ tal que los vectores transformados tengan **media 0 y covarianza identidad** (isótropos):
1. $\boldsymbol\mu=\frac1N\sum_i\mathbf x_i$ y $\Sigma=\frac1N\sum_i(\mathbf x_i-\boldsymbol\mu)^\top(\mathbf x_i-\boldsymbol\mu)$.
2. Descomposición espectral $\Sigma=U\Lambda U^\top$ ($U$ ortogonal, $\Lambda$ diagonal con varianzas $\lambda_1\ge\dots\ge\lambda_d$).
3. $W=U\Lambda^{-1/2}$. Comprobación: $\operatorname{Cov}(\tilde{\mathbf x}) = W^\top\Sigma W=\Lambda^{-1/2}U^\top U\Lambda U^\top U\Lambda^{-1/2}=I$.
4. **Reducción**: quedarse con las $k$ primeras columnas de $W$ (las direcciones de más varianza) reduce dimensión *y* ruido.

✍️ **Ejemplo a mano (2D, sin rotación).** Si $\Sigma = \mathrm{diag}(4, 0.25)$, entonces $U = I$, $\Lambda^{-1/2} = \mathrm{diag}(0.5, 2)$ y $W = \mathrm{diag}(0.5, 2)$. Un vector centrado $(2, 0.5)$, a una desviación típica en cada eje, pasa a $(1, 1)$: ambos ejes cuentan lo mismo. Antes, el coseno lo decidía casi solo el eje 1 (varianza 16 veces mayor): la anisotropía del módulo 09 en miniatura. Con $k = 1$ te quedarías solo con el eje 1 (reducción).

**Relación con BERT-flow** (Li et al., 2020): aprende un *normalizing flow* invertible que lleva los embeddings a una gaussiana isótropa; whitening es el caso **lineal** de esa idea y Su et al. reportan resultados comparables o mejores sin entrenar nada. **All-but-the-top** (Mu & Viswanath, 2018) es aún más simple: centrar y quitar las primeras componentes principales.
⚠️ $\boldsymbol\mu$ y $W$ se ajustan **con los documentos del índice** (o un corpus de referencia), nunca con las consultas de evaluación.

In [ ]:
ks = [8, 16, 32, 64, 128, 256]
fig, ax = plt.subplots(figsize=(7.5, 3.6))
post = {}
for (n, (D, Q)), col in zip(EMB.items(), [PAL["gris"], PAL["azul"], PAL["verde"]]):
    base = evalua(coseno(Q, D))["ndcg"]
    mu = D.mean(0, keepdims=True)
    cent = evalua(coseno(Q - mu, D - mu))["ndcg"]
    abtt = evalua(coseno(quita_top(Q, mu, 1), quita_top(D, mu, 1)))["ndcg"]
    kk = [k for k in ks if k <= D.shape[1]]
    wh = []
    for k in kk:
        m_, W = ajusta_whitening(D, k)
        wh.append(evalua(coseno((Q - m_) @ W, (D - m_) @ W))["ndcg"])
    post[n] = dict(base=base, centrado=cent, abtt=abtt, whitening=dict(zip(kk, wh)), aniso=anisotropia(D),
                   aniso_w=anisotropia((D - mu) @ ajusta_whitening(D)[1]))
    ax.plot(kk, wh, "o-", color=col, label=f"{n}: whitening-k")
    ax.axhline(base, ls=":", color=col)
ax.set_xscale("log", base=2); ax.set(xlabel="k (dimensiones tras whitening)", ylabel="nDCG@10", title="Whitening con reducción a k dims (punteado: sin post-procesado)")
ax.legend(fontsize=7); plt.tight_layout(); plt.show()
for n, r_ in post.items():
    best_k = max(r_["whitening"], key=r_["whitening"].get)
    print(f"{n:30s} base={r_['base']:.3f} | centrado={r_['centrado']:.3f} | ABTT-1={r_['abtt']:.3f} | whitening mejor k={best_k} (elegido en test: optimista): {r_['whitening'][best_k]:.3f} | anisotropía {r_['aniso']:.2f} → {r_['aniso_w']:.2f}")

> 👀 **Qué debes observar.** (1) Tras el whitening completo la anisotropía cae a ≈ 0 por construcción. (2) Que eso **mejore el retrieval** no está garantizado: el whitening iguala *todas* las direcciones, también las de ruido. Con $k$ pequeño se pierde información; con $k$ grande se amplifica ruido; suele haber un óptimo intermedio, y debe elegirse con un conjunto de validación, no con el test.
> (3) En el *fallback* el efecto depende del encoder y de la semilla: en la ejecución de referencia (oct-2026, CPU) el whitening llevó el nDCG@10 del mini-SBERT de 0.37 a 0.51 ($k=64$), pero con el mini-BERT crudo **empeoró** (0.21 → 0.19), y en versiones anteriores del fallback (vocabulario WordPiece no determinista) el signo del efecto llegó a invertirse entre ejecuciones. Ahora el fallback es reproducible (vocabulario determinista + semillas fijas: mismas cifras al re-ejecutar), pero con otra semilla o con datos reales tus números cambiarán. La lección es justo esa: un post-procesado no crea señal que el encoder no tiene, y su efecto hay que **validarlo**, no suponerlo. Su et al. (2021) reportan mejoras grandes sobre BERT crudo en STS; para encoders bien entrenados contrastivamente la ganancia suele ser pequeña o negativa: mídelo con tus modelos.

> ### 🎨 GEMINI-IMAGEN · ID: `M10-IMG-03` · ⚠️ REEMPLAZAR ESTA CELDA
> **Anisotropía: del cono a la esfera (centrado y whitening)**
>
> **Qué muestra:** Visualiza el efecto geométrico de centrar y de blanquear un conjunto de embeddings anisótropos, conectando con la caída del coseno medio.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Tres paneles: nube anisótropa en un cono con su media, la nube centrada (elipse) y la nube blanqueada (círculo).
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, 16:9, tres paneles de izquierda a derecha con flechas entre ellos. Paleta: azul #1F6FEB para los puntos, naranja #F28E2B para la dirección media, gris #6B7280 para ejes y texto.
> Panel 1, título «Anisótropo»: ejes 2D con el origen marcado; una nube alargada y estrecha de 40 puntos azules lejos del origen, todos dentro de un cono delgado; una flecha naranja gruesa desde el origen hasta el centro de la nube rotulada «μ (dirección media)». Texto bajo el panel: «coseno medio ≈ 0.8».
> Flecha a la derecha con el rótulo «restar μ».
> Panel 2, título «Centrado»: la misma nube, ahora centrada en el origen pero todavía alargada en diagonal (elipse). Texto: «coseno medio ≈ 0, pero varianza desigual».
> Flecha a la derecha con el rótulo «× U Λ^(-1/2)».
> Panel 3, título «Whitening»: nube redonda (círculo) centrada en el origen. Texto: «covarianza = I (isótropo)».
> Franja inferior: «Whitening = (x − μ) · U Λ^(-1/2); con k columnas también reduce dimensión».
> Evita: texto ilegible, flechas ambiguas, añadir números distintos de los indicados, decoración.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M10-IMG-03.png` en la carpeta del módulo y reemplaza esta celda por: `![Tres paneles: nube anisótropa en un cono con su media, la nube centrada (elipse) y la nube blanqueada (círculo).](assets/M10-IMG-03.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> [cono estrecho lejos del origen] --restar μ--> [elipse centrada] --×UΛ^-1/2--> [círculo centrado]
>   cos medio ≈ 0.8                          cos medio ≈ 0               covarianza = I
> ```

> 👀 **Qué debes observar.** El centrado arregla el *coseno medio*; el whitening además iguala la varianza por dirección. Son transformaciones lineales que se aplican igual a consultas y documentos y se pueden «hornear» en el índice.

### Idioma como dirección dominante
En un encoder multilingüe, una parte de la varianza codifica **qué idioma** es el texto, no de qué trata. Libovický et al. (2019) mostraron que restar la **media por idioma** hace las representaciones de mBERT más «neutras» respecto al idioma. Lo comprobamos visualmente en Políglota (ES, EN y código).

In [ ]:
langs = np.array([d["lang"] for d in docs])
n_sel = list(EMB)[-1]                                  # el encoder ajustado para similitud (último de la lista)
D, Q = EMB[n_sel]
D_lc = D.copy()
for l in np.unique(langs):
    D_lc[langs == l] -= D[langs == l].mean(0)


def pca2(X):
    Xc = X - X.mean(0)
    _, _, Vt = np.linalg.svd(Xc, full_matrices=False)
    return Xc @ Vt[:2].T


fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.8))
for a, X, tit in ((ax[0], D, "original"), (ax[1], D_lc, "tras restar la media de cada idioma")):
    P = pca2(X)
    for l, col in zip(["es", "en", "code"], [PAL["azul"], PAL["naranja"], PAL["verde"]]):
        a.scatter(P[langs == l, 0], P[langs == l, 1], s=8, alpha=.6, color=col, label=l)
    a.set(title=f"{n_sel}: PCA ({tit})", xlabel="PC1", ylabel="PC2"); a.legend()
plt.tight_layout(); plt.show()
print("silhouette por idioma (↑ = idiomas más separados):", end=" ")
from sklearn.metrics import silhouette_score
print(f"original={silhouette_score(D, langs, metric='cosine'):.3f} | centrado por idioma={silhouette_score(D_lc, langs, metric='cosine'):.3f}")

> 👀 **Qué debes observar.** En el panel izquierdo los documentos se agrupan **por idioma** (sobre todo el código frente a la prosa): las primeras componentes principales codifican la lengua. Tras centrar por idioma, el *silhouette* por idioma baja. Para búsqueda **cruzada** (consulta en ES, documento en EN) eso ayuda; para búsqueda monolingüe puede dar igual o empeorar. Ojo: para centrar la consulta necesitas conocer (o detectar) su idioma.

## 7. Coseno, producto escalar y L2
Para vectores $\mathbf q,\mathbf d$: $\;\|\mathbf q-\mathbf d\|^2=\|\mathbf q\|^2+\|\mathbf d\|^2-2\,\mathbf q\cdot\mathbf d$.
- Si **todos** los vectores tienen norma 1: $\|\mathbf q-\mathbf d\|^2 = 2-2\cos(\mathbf q,\mathbf d)$, así que **coseno, producto escalar y L2 dan el mismo ranking**.
- Sin normalizar, para una consulta fija $\|\mathbf q\|$ es constante, pero $\|\mathbf d\|^2$ no: L2 penaliza documentos de norma grande y el producto escalar los **premia**. La norma pasa a ser una *feature* (en recsys suele correlacionar con popularidad; Steck et al., 2024, discuten cuándo el coseno sobre embeddings aprendidos no mide lo que creemos).
- Regla práctica: usa la métrica **con la que se entrenó** el modelo (DPR: producto escalar; SBERT/E5: coseno) y, si normalizas, guarda los vectores normalizados en el índice (módulo 14).

In [ ]:
from scipy.stats import kendalltau

D, Q = EMB[n_sel]
metricas = {"coseno": coseno(Q, D), "producto": Q @ D.T, "−L2": -((Q ** 2).sum(1)[:, None] + (D ** 2).sum(1)[None] - 2 * Q @ D.T)}
Dn, Qn = D / np.linalg.norm(D, axis=1, keepdims=True), Q / np.linalg.norm(Q, axis=1, keepdims=True)
metricas["producto (normalizado)"] = Qn @ Dn.T
nombres = list(metricas)
tau = np.zeros((len(nombres), len(nombres)))
for a_, na in enumerate(nombres):
    for b_, nb_ in enumerate(nombres):
        tau[a_, b_] = np.mean([kendalltau(metricas[na][i], metricas[nb_][i]).statistic for i in range(0, len(Q), 3)])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
im = ax[0].imshow(tau, vmin=0, vmax=1, cmap="Blues")
ax[0].set_xticks(range(len(nombres)), nombres, rotation=30, ha="right"); ax[0].set_yticks(range(len(nombres)), nombres)
for i in range(len(nombres)):
    for j in range(len(nombres)):
        ax[0].text(j, i, f"{tau[i, j]:.2f}", ha="center", va="center", color="k" if tau[i, j] < .7 else "w")
ax[0].set_title("τ de Kendall medio entre rankings"); plt.colorbar(im, ax=ax[0])
ax[1].hist(np.linalg.norm(D, axis=1), bins=40, color=PAL["gris"]); ax[1].set(xlabel="‖d‖", ylabel="nº de documentos", title="Normas de los documentos")
plt.tight_layout(); plt.show()
for k_, S in metricas.items():
    print(f"{k_:24s} nDCG@10={evalua(S)['ndcg']:.3f}")

> 👀 **Qué debes observar.** «coseno» y «producto (normalizado)» tienen τ = 1.00: son el mismo ranking. Producto escalar y −L2 sin normalizar se separan tanto más cuanto más varían las normas de los documentos (histograma derecho). Si tu modelo no normaliza y las normas varían mucho, la elección de métrica cambia resultados: compruébalo antes de elegir el tipo de índice.

## 8. Prefijos e instrucciones
Un bi-encoder **simétrico** (misma torre, sin marcas) trata igual «consulta» y «documento». Pero una consulta corta («caché redis timeout») y un párrafo no son el mismo tipo de texto. E5 (Wang et al., 2022) entrena con prefijos `query: ` y `passage: ` («rompen la simetría»; el paper dice que importa sobre todo cuando el corpus contiene paráfrasis de la consulta). INSTRUCTOR (Su et al., 2022) generaliza la idea a **instrucciones** en lenguaje natural por tarea, y los embedders basados en LLMs del módulo 12 las usan de forma sistemática.
Regla de oro: **usa exactamente los prefijos con los que se entrenó el modelo**; omitirlos o inventarlos es una de las causas más comunes de «el modelo funciona peor que en el leaderboard».

In [ ]:
# Ablación de prefijos: solo tiene sentido con E5 real; con el fallback se ejecuta igual pero el modelo nunca vio prefijos.
n_pref = next((n for n, e in encs.items() if e.prefijos != ("", "")), list(encs)[-1])
e = encs[n_pref]
variantes = {"query:/passage:": ("query: ", "passage: "), "sin prefijos": ("", ""), "query: en ambos lados": ("query: ", "query: ")}
orig = e.prefijos
if orig == ("", ""):
    print(f"(nota: {n_pref} NO se entrenó con prefijos; las diferencias aquí son ruido, no un efecto)")
for nombre, pre in variantes.items():
    e.prefijos = pre
    S = coseno(e.encode(q_txt, "mean", tipo="query"), e.encode(doc_txt, "mean", tipo="doc"))
    print(f"{n_pref} · {nombre:34s} nDCG@10={evalua(S)['ndcg']:.3f}")
e.prefijos = orig

> 👀 **Qué debes observar.** Con E5 real, quitar los prefijos o usar `query:` en ambos lados cambia el nDCG@10: el modelo aprendió a tratar distinto cada lado. Con el *fallback* (sin prefijos en su entrenamiento) las diferencias son ruido: ese contraste es justo la lección, un prefijo solo significa algo si el modelo se entrenó con él.

## 9. Bi-encoder vs cross-encoder
- **Bi-encoder** (SBERT): $\text{score}(q,d)=\cos(f(q),f(d))$. Los documentos se codifican **una vez** y se indexan; una consulta cuesta 1 forward + una búsqueda ANN. Coste de buscar en $N$ documentos: $O(1)$ forwards por consulta.
- **Cross-encoder**: $\text{score}(q,d)=g([q;d])$, un Transformer que lee consulta y documento juntos (atención cruzada completa). Más preciso, pero cuesta **$N$ forwards por consulta**: inviable como primera etapa, ideal para **re-rankear** los top-100 (módulo 13).
- SBERT (Reimers & Gurevych, 2019) mostró que entrenar el bi-encoder siamés (pesos compartidos) para similitud resolvía el problema de BERT crudo: es el patrón que entrenarás tú en el módulo 11.

> ### 🎨 GEMINI-IMAGEN · ID: `M10-IMG-04` · ⚠️ REEMPLAZAR ESTA CELDA
> **Bi-encoder (siamés) frente a cross-encoder**
>
> **Qué muestra:** Compara las dos arquitecturas de similitud y su coste: dos torres con pesos compartidos y coseno vs un único Transformer que lee el par concatenado.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Bi-encoder con dos torres de pesos compartidos y coseno frente a cross-encoder con un único Transformer sobre el par concatenado.
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, 16:9, dos mitades separadas por una línea vertical gris. Paleta: azul #1F6FEB (bi-encoder), naranja #F28E2B (cross-encoder), verde #2CA02C (salidas), gris #6B7280 (texto secundario).
> Mitad izquierda, título «Bi-encoder (SBERT)»: dos cajas de texto abajo «consulta: cómo configurar la caché» y «documento: La caché es una memoria rápida…». Encima de cada una, una torre azul idéntica rotulada «Transformer + pooling» y entre las dos torres un candado con el texto «pesos compartidos». Sobre cada torre un vector verde «u» y «v», y arriba una caja «cos(u, v)». A un lado: «documentos: se codifican UNA vez y se indexan (ANN)» y «coste por consulta: 1 forward + búsqueda».
> Mitad derecha, título «Cross-encoder»: una sola caja de texto «[CLS] consulta [SEP] documento [SEP]», encima una única torre naranja más ancha rotulada «Transformer (atención cruzada completa)» y arriba una caja verde «score ∈ ℝ». A un lado: «coste por consulta: N forwards (uno por documento)» y «uso típico: re-rankear el top-100».
> Evita: texto ilegible, flechas ambiguas, números inventados, decoración.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M10-IMG-04.png` en la carpeta del módulo y reemplaza esta celda por: `![Bi-encoder con dos torres de pesos compartidos y coseno frente a cross-encoder con un único Transformer sobre el par concatenado.](assets/M10-IMG-04.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> BI-ENCODER                          | CROSS-ENCODER
> q ─► [Transformer+pool] ─► u ┐      | [CLS] q [SEP] d [SEP] ─► [Transformer] ─► score
> d ─► [Transformer+pool] ─► v ┴ cos  | coste: N forwards/consulta → re-ranking
> (pesos compartidos; d se indexa)    |
> ```

> 👀 **Qué debes observar.** La única diferencia estructural es **dónde** se mezclan consulta y documento: al final (un coseno, factorizable e indexable) o desde la primera capa (atención cruzada, no indexable). Es el mismo *trade-off* que *two-tower* vs *ranker* en recsys.

## Mapa de madurez (a oct-2026)

| Técnica | Madurez | Quién la usa (verificable) | Fuente |
|---|---|---|---|
| Bi-encoder siamés + mean pooling | 🟢 sólido | sentence-transformers (SBERT), E5 (*average pooling*) | Reimers & Gurevych 2019; Wang et al. 2022 |
| CLS pooling | 🟢 sólido (según modelo) | DPR usa la representación de `[CLS]` | Karpukhin et al. 2020 |
| Last-token pooling en decoders | 🟡 emergente → estándar en LLM-embedders | SGPT; embedders basados en LLMs (módulo 12) | Muennighoff 2022 |
| Prefijos / instrucciones | 🟢 sólido | E5 (`query:`/`passage:`), INSTRUCTOR | Wang et al. 2022; Su et al. 2022 |
| Whitening / centrado post-hoc | 🟡 útil para encoders crudos; poco usado con embedders modernos | — | Su et al. 2021; Mu & Viswanath 2018 |
| BERT-flow | 🔴 mayormente superado (whitening lineal comparable) | — | Li et al. 2020; Su et al. 2021 |
| Multilingüe por destilación | 🟢 sólido | `paraphrase-multilingual-*` de sentence-transformers | Reimers & Gurevych 2020 |
| Cross-encoder para re-ranking | 🟢 sólido | ver módulo 13 | Reimers & Gurevych 2019 |

## 🏭 En producción
- **El pooling viaja con el modelo.** En sentence-transformers el módulo `Pooling` se guarda en la configuración del modelo; si reimplementas la inferencia (ONNX, TEI, tu propio servidor), verifica numéricamente contra la librería como hicimos arriba (error ~1e-6).
- **Prefijos en producción.** Los modelos tipo E5 requieren `query: `/`passage: `; muchos embedders recientes usan instrucciones por tarea (módulo 12). Un servicio que indexa con un prefijo y consulta con otro degrada en silencio: inclúyelo en tests de integración.
- **Normaliza una vez.** Si el modelo se entrenó con coseno, guarda vectores L2-normalizados y usa producto escalar en el índice (FAISS `IndexFlatIP`, HNSW con IP): más rápido y equivalente (módulo 14).
- **Post-procesado como transformación del índice.** Centrado/whitening son una matriz $d\times k$: se aplican al indexar y al consultar, se versionan junto al modelo y, con $k<d$, reducen memoria.
- **Para Políglota:** el bi-encoder será la primera etapa (recall), con BM25 en híbrido (13) y un cross-encoder opcional para re-rankear.

## 🧠 Secretos de la élite
1. **Embeddings crudos de BERT ≠ embeddings de similitud.** SBERT midió que la media de BERT sin ajustar (54.81 de media en STS) y su CLS (29.19) quedaban por debajo de promediar GloVe (61.32). Antes de culpar al índice, comprueba que el modelo fue entrenado para similitud.
2. **Audita el pooling contra la librería.** Una diferencia de pooling (CLS vs mean, o media sin máscara) puede hundir el nDCG sin dar ningún error; nuestra comprobación numérica tarda segundos.
3. **`h[:, -1]` no es el último token** con *padding* a la derecha: usa la máscara (como `ultimo_token`). Es un bug clásico al convertir LLMs en embedders.
4. **El whitening se ajusta con documentos, y $k$ se elige en validación.** Ajustarlo con las consultas de test es *leakage*; y su efecto depende del encoder (en nuestro fallback: +0.14 de nDCG@10 con el mini-SBERT y −0.02 con el mini-BERT crudo).
5. **El idioma es una dirección.** En multilingüe, centrar por idioma reduce la separación por lengua (lo mediste con silhouette); útil para búsqueda cruzada, pero exige saber el idioma de la consulta.
6. **Coseno = producto con vectores normalizados (τ = 1.00).** Si la norma varía, el producto escalar introduce un sesgo hacia documentos de norma grande, análogo al sesgo de popularidad en recsys.
7. **Respeta los prefijos del entrenamiento.** Es la primera comprobación cuando un embedder rinde menos que en MTEB.

## ⚠️ Errores comunes
- Media sin máscara de *padding* (embeddings que dependen del lote).
- Usar CLS en un modelo entrenado con media (o al revés).
- Ajustar whitening/centrado con el conjunto de test.
- Mezclar métrica: entrenado con coseno, indexado con producto escalar sin normalizar.
- Olvidar los prefijos `query:`/`passage:` o aplicarlos en un modelo que no los usa.
- Truncar documentos largos sin darse cuenta (`max_length`): el final del documento no existe para el modelo.
- Concluir que «los embeddings no funcionan» comparando con BM25 en un benchmark de ítem conocido con fuerte solapamiento léxico.

## Autoevaluación
<details><summary><b>1. (Cálculo)</b> Dos vectores unitarios tienen coseno 0.8. ¿Cuál es su distancia L2? ¿Y si el coseno es 0.2?</summary>

$\|\mathbf q-\mathbf d\|=\sqrt{2-2\cdot0.8}=\sqrt{0.4}\approx0.632$; con 0.2: $\sqrt{1.6}\approx1.265$.
</details>

<details><summary><b>2. (Cálculo)</b> Un cross-encoder tarda 5 ms por par en GPU. ¿Cuánto tarda en puntuar 1 M de documentos para una consulta? ¿Y re-rankear el top-100?</summary>

$10^6\times5$ ms $=5000$ s ≈ 83 min por consulta; el top-100, 0.5 s (y en lote, menos). Por eso el cross-encoder va **después** del bi-encoder.
</details>

<details><summary><b>3. (Diagnóstico)</b> Tu servicio da nDCG 0.42 y el mismo modelo en tu notebook 0.55. El código de pooling es «el mismo». ¿Qué tres cosas compruebas?</summary>

(a) Máscara de padding en la media (el servicio agrupa en lotes de tamaño distinto); (b) prefijos/instrucciones idénticos al indexar y al consultar; (c) normalización y métrica del índice (coseno vs producto sin normalizar), y de paso `max_length`.
</details>

<details><summary><b>4. (Diagnóstico)</b> Tras whitening completo ($k=d$), el coseno medio entre documentos es ≈ 0 pero el nDCG bajó. ¿Contradicción?</summary>

No. Isotropía no es relevancia: el whitening da el mismo peso a direcciones de ruido que a las informativas. Reduce $k$ (quedarte con las direcciones de mayor varianza) y elige $k$ en validación, o usa solo centrado.
</details>

<details><summary><b>5. (Transferencia)</b> En tu *two-tower* de recsys, los ítems populares tienen normas grandes. ¿Coseno o producto escalar para recuperar candidatos? ¿Qué cambia?</summary>

Con producto escalar la norma actúa como prior de popularidad (más clics, menos diversidad); con coseno se elimina. La elección depende del objetivo; lo correcto es usar la del entrenamiento y, si quieres controlar popularidad, hacerlo explícitamente (logQ, re-ranking).
</details>

<details><summary><b>6.</b> ¿Por qué la self-similarity de una palabra en la capa 0 es casi 1?</summary>

Porque la capa 0 es el embedding de entrada: la misma fila de $E$ (más la posición) en todos los contextos.
</details>

<details><summary><b>7. (Transferencia)</b> Políglota recibe consultas en ES y documentos en EN. ¿Qué técnica de este módulo probarías primero sin reentrenar y cómo la validarías?</summary>

Centrado por idioma (restar la media de documentos de cada idioma y la de las consultas de su idioma detectado) y medir nDCG@10 en un conjunto de validación **cruzado** (consulta ES → documento EN), comparando con el modelo sin centrar.
</details>

## Referencias
- Devlin, J., Chang, M.-W., Lee, K., & Toutanova, K. (2019). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding.* [arXiv:1810.04805](https://arxiv.org/abs/1810.04805)
- Reimers, N., & Gurevych, I. (2019). *Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks.* EMNLP. [arXiv:1908.10084](https://arxiv.org/abs/1908.10084)
- Reimers, N., & Gurevych, I. (2020). *Making Monolingual Sentence Embeddings Multilingual using Knowledge Distillation.* [arXiv:2004.09813](https://arxiv.org/abs/2004.09813)
- Ethayarajh, K. (2019). *How Contextual are Contextualized Word Representations? Comparing the Geometry of BERT, ELMo, and GPT-2 Embeddings.* EMNLP. [arXiv:1909.00512](https://arxiv.org/abs/1909.00512)
- Li, B., Zhou, H., He, J., et al. (2020). *On the Sentence Embeddings from Pre-trained Language Models* (BERT-flow). [arXiv:2011.05864](https://arxiv.org/abs/2011.05864)
- Su, J., Cao, J., Liu, W., et al. (2021). *Whitening Sentence Representations for Better Semantics and Faster Retrieval.* [arXiv:2103.15316](https://arxiv.org/abs/2103.15316)
- Mu, J., Bhat, S., & Viswanath, P. (2018). *All-but-the-Top: Simple and Effective Postprocessing for Word Representations.* ICLR. [arXiv:1702.01417](https://arxiv.org/abs/1702.01417)
- Timkey, W., & van Schijndel, M. (2021). *All Bark and No Bite: Rogue Dimensions in Transformer Language Models Obscure Representational Quality.* [arXiv:2109.04404](https://arxiv.org/abs/2109.04404)
- Godey, N., de la Clergerie, É., & Sagot, B. (2024). *Anisotropy Is Inherent to Self-Attention in Transformers.* [arXiv:2401.12143](https://arxiv.org/abs/2401.12143)
- Libovický, J., Rosa, R., & Fraser, A. (2019). *How Language-Neutral is Multilingual BERT?* [arXiv:1911.03310](https://arxiv.org/abs/1911.03310)
- Wang, L., Yang, N., Huang, X., et al. (2022). *Text Embeddings by Weakly-Supervised Contrastive Pre-training* (E5). [arXiv:2212.03533](https://arxiv.org/abs/2212.03533)
- Su, H., Shi, W., Kasai, J., et al. (2022). *One Embedder, Any Task: Instruction-Finetuned Text Embeddings* (INSTRUCTOR). [arXiv:2212.09741](https://arxiv.org/abs/2212.09741)
- Muennighoff, N. (2022). *SGPT: GPT Sentence Embeddings for Semantic Search.* [arXiv:2202.08904](https://arxiv.org/abs/2202.08904)
- Karpukhin, V., Oğuz, B., Min, S., et al. (2020). *Dense Passage Retrieval for Open-Domain Question Answering.* [arXiv:2004.04906](https://arxiv.org/abs/2004.04906)
- Steck, H., Ekanadham, C., & Kallus, N. (2024). *Is Cosine-Similarity of Embeddings Really About Similarity?* [arXiv:2403.05440](https://arxiv.org/abs/2403.05440)
- Librería: sentence-transformers (https://pypi.org/project/sentence-transformers/), módulos `Transformer` y `Pooling`.

## Siguiente módulo
**11 · Aprendizaje contrastivo**: entrenar tú el bi-encoder de Políglota con InfoNCE, negativos en el lote y *hard negatives*, desde cero en PyTorch y con sentence-transformers.